### We need to consider the different 'inputs' we can tweak.
This includes:
1) Activity of methanogens and methanotrophs (as a function of pH, water (rain), sulfate, temperature, and fluctuations)

Assumptions:
1) The biomass is stored dry, in a mountain-like pile.
2) Oxygen is zero, as it has already been used up. The volume of the pile that is exposed to the air is very small compared to the total volume.

PROGRESS:
1) So far, just a duplicate of https://docs.google.com/spreadsheets/d/1j90kZKRXXC5wFWEThdobLPqA5XyLCwp26xlm8BnWfiI/edit?gid=0#gid=0

IDEAS:
* pcr around methanogen genes, might be tricky, could be interesting!
* use astropi for the units everywhere
* create a seperate directory for it later, pull all of the functions into .py files, and then import them as needed into the notebook
* use bokeh (also for interactive diagrams, buttons, sliders)

In [1]:
# !pip install --upgrade ipywidgets
import ipywidgets as widgets
from IPython.display import display

import math
from math import log

import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
# Toggle between Linear and Exponential
decay_widget = widgets.ToggleButtons(
    options=['Linear', 'Exponential'],
    description='Decay model:',
    value='Linear',
    button_style='info'
)

# Percent of carbon released after 100 years
pct_widget = widgets.FloatText(
    value=0.01,
    description='% released in 100 y:',
    step=0.001
)

display(decay_widget, pct_widget)


ToggleButtons(button_style='info', description='Decay model:', options=('Linear', 'Exponential'), value='Linea…

FloatText(value=0.01, description='% released in 100 y:', step=0.001)

In [3]:
out = widgets.Output()

def setup_linear():
    clear_output()

    # inputs for starting amounts
    biomass_w = widgets.FloatText(description='Starting mg biomass:', value=1750.0, step=0.1)
    carbon_w  = widgets.FloatText(description='Starting mg carbon:',  value=875.0,  step=0.1)
    run_btn   = widgets.Button(description='Compute', button_style='success')

    display(biomass_w, carbon_w, run_btn, out)

    def on_run(b):
        with out:
            out.clear_output()
            start_carbon = carbon_w.value
            pct_100y     = pct_widget.value      # from cell 1

            # linear decay percentages
            pct_per_year  = pct_100y / 100           
            pct_per_month = pct_per_year / 12
            pct_per_week = pct_per_year / 52           

            # calculations
            carbon_mg_month = start_carbon * pct_per_month
            moles_C   = carbon_mg_month / 1000 / 12.011
            liters_CO2 = moles_C * 22.414
            ml_CO2     = liters_CO2 * 1000
            pct_40ml   = ml_CO2 / 40 * 100
            ppm_in_1m3 = pct_40ml * 10000           

            # printing everything
            print(f"Working with the assumption that {pct_100y:.2f}")
            print(f"% lost in 1 yr: {pct_per_year:.6f} %")
            print(f"% lost in 1 month: {pct_per_month:.6f} %")
            print(f"% lost in 1 week: {pct_per_month:.6f} %")
            print(f"Carbon used per month: {carbon_mg_month:.6f} mg C")
            print(f"Moles C per month: {moles_C:.6e} mol")
            print(f"CO₂ volume at STP: {liters_CO2:.6f} L  ({ml_CO2:.6f} mL)")
            print(f"{pct_40ml:.6f} % of a 40 mL headspace")
            print(f"≈ {ppm_in_1m3:.2f} ppm in 1 m³ air")

    run_btn.on_click(on_run)


In [4]:
def setup_exponential():
    clear_output()

    # Inputs for starting amounts
    biomass_w = widgets.FloatText(description='Starting mg biomass:', value=1750.0, step=0.1)
    carbon_w  = widgets.FloatText(description='Starting mg carbon:',  value=875.0,  step=0.1)
    run_btn   = widgets.Button(description='Compute', button_style='success')

    display(biomass_w, carbon_w, run_btn, out)

    def on_run(b):
        with out:
            out.clear_output()
            start_carbon = carbon_w.value
            pct_100y     = pct_widget.value      # from cell 1

            # Decay calculations
            decay_exp = math.log((100 - pct_100y) / 100) / -1200
            pct_first_year  = 100-(100*math.exp(-decay_exp*12))           # % lost first year
            pct_first_month = 100-(100*math.exp(-decay_exp*1)) 
            pct_first_week = 100-(100*math.exp(-decay_exp*0.23))         # % lost first month

            # calculations
            carbon_mg_firstmonth = start_carbon * pct_first_month
            carbon_mg_firstweek = start_carbon * pct_first_week
            moles_C   = carbon_mg_firstmonth / 1000 / 12.011
            liters_CO2 = moles_C * 22.414
            ml_CO2     = liters_CO2 * 1000
            pct_40ml   = ml_CO2 / 40 * 100
            ppm_in_1m3 = pct_40ml * 10000           # 1 m³ ≈ 1000 L

            # printing everything
            print(f"Working with the assumption that {pct_100y:.2f} is released in `100 years...")
            print(f"% lost in first yr: {pct_first_year:.6f} %")
            print(f"% lost in first month: {pct_first_month:.6f} %")
            print(f"% lost in first week: {pct_first_week:.6f} %")
            print(f"Carbon used first month: {carbon_mg_firstmonth:.6f} mg C")
            print(f"Carbon used first week: {carbon_mg_firstweek:.6f} mg C")
            print(f"Moles C per month: {moles_C:.6e} mol")
            print(f"CO₂ volume at STP: {liters_CO2:.6f} L  ({ml_CO2:.6f} mL)")
            print(f"{pct_40ml:.6f} % of a 40 mL headspace")
            print(f"≈ {ppm_in_1m3:.2f} ppm in 1 m³ air")

    run_btn.on_click(on_run)



In [ ]:
# output depends on whether selected linear or exponential decay
if decay_widget.value == 'Linear':
    setup_linear()
else:

    print("Working on exponential.")
    setup_exponential()

FloatText(value=1750.0, description='Starting mg biomass:', step=0.1)

FloatText(value=875.0, description='Starting mg carbon:', step=0.1)

Button(button_style='success', description='Compute', style=ButtonStyle())

Output()